In [20]:
import json
import tiktoken


# Initialize the tiktoken encoding.
# For this example, we're using the GPT-2 encoding.
# You may adjust this if Mistral uses a different tokenization scheme.
encoding = tiktoken.get_encoding("gpt2")

def count_tokens(text):
    """Counts the number of tokens in a given text using tiktoken."""
    return len(encoding.encode(text))


In [19]:
!pip3 install tiktoken



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
  Using cached tiktoken-0.9.0-cp39-cp39-macosx_11_0_arm64.whl (1.0 MB)
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
# Define the input JSON Lines files
input_files = ["/Users/abeltran/Documents/wmca/scripts/evaluation_docs_extracted.jsonl", 
               "/Users/abeltran/Documents/wmca/scripts/government_docs_extracted.jsonl"]

# Load all records from both files into one list
data = []
for file in input_files:
    with open(file, "r") as infile:
        for line in infile:
            if line.strip():  # skip empty lines
                try:
                    record = json.loads(line)
                    data.append(record)
                except json.JSONDecodeError:
                    print(f"Skipping malformed line in {file}")

print(f"Loaded a total of {len(data)} records from {', '.join(input_files)}")

In [28]:
# Iterate through each record and count tokens for each section.
for record in data:
    filename = record.get("filename", "unknown_file")
    sections = record.get("sections", {})
    print(f"\nFilename: {filename}")
    for section_name, text in sections.items():
        tokens = count_tokens(text)
        print(f"  Section '{section_name}' has {tokens} tokens")



Filename: [11]wmca-saf-fbc-v3-2 CW Gigapark Sep 2024.md

Filename: [14]20230123 HUG2 BJC Observation Report v1.1.md
  Section 'PROJECT EXECUTIVE SUMMARY' has 1636 tokens
  Section 'PROJECT MATURITY ASSESSMENT' has 919 tokens
  Section 'FINDINGS' has 1669 tokens
  Section 'RECOMMENDATIONS' has 1513 tokens
  Section 'RAG STATUS DEFINITIONS' has 720 tokens

Filename: [12]20230130 Final BCAT Assurance Observations Report Social Housing v1.0.md
  Section 'PROJECT EXECUTIVE SUMMARY' has 1162 tokens
  Section 'PROJECT MATURITY ASSESSMENT' has 626 tokens
  Section 'FINDINGS' has 772 tokens
  Section 'RECOMMENDATIONS' has 715 tokens
  Section 'RAG STATUS DEFINITIONS' has 663 tokens

Filename: [13]PBC ATF3 BCAT Assurance Observations Report v.1.0 010622 Cyc and Walk Response.md
  Section 'PROJECT EXECUTIVE SUMMARY' has 870 tokens
  Section 'PROJECT MATURITY ASSESSMENT' has 1042 tokens
  Section 'FINDINGS' has 1906 tokens
  Section 'RECOMMENDATIONS' has 2267 tokens
  Section 'RAG STATUS DEFINITI

In [34]:
def chunk_text(text, max_tokens=1000, overlap=50):
    """
    Splits the text into chunks of at most max_tokens tokens, using tiktoken for tokenization.
    An overlap between chunks is added to preserve context.

    Args:
        text (str): The text to be chunked.
        max_tokens (int): Maximum tokens per chunk.
        overlap (int): Number of tokens to overlap between consecutive chunks.

    Returns:
        list: A list of text chunks.
    """
    # Encode the text to get token IDs
    tokens = encoding.encode(text)
    chunks = []
    start = 0
    
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        # Decode the tokens back to text for this chunk
        chunk = encoding.decode(tokens[start:end])
        chunks.append(chunk)
        
        # If we're at the end, break out
        if end == len(tokens):
            break
        
        # Slide the window by subtracting the overlap tokens
        start = end - overlap
    
    return chunks

# Example usage: chunk a long section
example_text = data[50]["sections"].get("STRATEGIC CASE", "")
chunks = chunk_text(example_text, max_tokens=1000, overlap=50)
print(f"Chunked into {len(chunks)} parts.")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} token count: {count_tokens(chunk)}")


Chunked into 6 parts.
Chunk 1 token count: 1000
Chunk 2 token count: 1000
Chunk 3 token count: 1000
Chunk 4 token count: 1000
Chunk 5 token count: 1000
Chunk 6 token count: 780


In [ ]:
!python3 -c "import platform; print(platform.processor())"


arm


In [1]:
!pip3 install mlx_lm



Defaulting to user installation because normal site-packages is not writeable
  Using cached mlx_lm-0.22.2-py3-none-any.whl (162 kB)
  Using cached protobuf-6.30.1-cp39-abi3-macosx_10_9_universal2.whl (417 kB)
     |████████████████████████████████| 27.9 MB 1.5 MB/s eta 0:00:01
  Using cached transformers-4.50.0-py3-none-any.whl (10.2 MB)
     |████████████████████████████████| 172 kB 13.4 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 30.4 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 11.4 MB/s eta 0:00:01
     |████████████████████████████████| 284 kB 2.4 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 5.3 MB/s  eta 0:00:01
  Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl (418 kB)
  Using cached huggingface_hub-0.29.3-py3-none-any.whl (468 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-macosx_11_0_arm64.whl (2.7 MB)
     |████████████████████████████████| 64 kB 4.2 MB/s  eta 0:00:01
  Using cached sentencepiece-0.2.0-c

In [ ]:
from mlx_lm import generate, load

# Load the Mistral model (adjust path/configuration accordingly)
checkpoint = "scripts/llm/mlx_models/Mistral-7B-Instruct-v0.3"
# Load the corresponding model and tokenizer

# Load the model and tokenizer
model, tokenizer = load(path_or_hf_repo=checkpoint)

# Specify the prompt and conversation history
prompt = "Why is the sky blue?"
conversation = [{"role": "user", "content": prompt}]

# Transform the prompt into the chat template
prompt = tokenizer.apply_chat_template(
    conversation=conversation, add_generation_prompt=True
)

# Specify the maximum number of tokens
max_gen_tokens = 2048

# Specify if tokens and timing information will be printed
verbose = True

# Generate a response with the specified settings
response = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_tokens=max_gen_tokens,
    verbose=verbose,
)


The sky appears blue due to a process called Rayleigh scattering. As sunlight reaches Earth, it is made up of different colors, which are essentially different wavelengths of light. Shorter wavelengths (like blue and violet) are scattered in all directions more than longer wavelengths (like red, yellow, and green). However, we see the sky as blue, not violet, because our eyes are more sensitive to blue light and because sunlight reaches us with less violet light (it gets absorbed by the atmosphere more) than blue light. Additionally, the human eye is more sensitive to blue light, and our brain interprets the combined blue and violet light as blue.
Prompt: 9 tokens, 9.990 tokens-per-sec
Generation: 143 tokens, 28.469 tokens-per-sec
Peak memory: 12.501 GB


In [54]:
def summarize_chunk(chunk, model, tokenizer, max_tokens=max_gen_tokens, verbose=verbose):
    """
    Summarize a given text chunk using the Mistral model via mlx-lm.
    
    Args:
        chunk (str): The text chunk to summarize.
        model: The loaded Mistral model.
        tokenizer: The tokenizer for the Mistral model.
        max_tokens (int): Maximum tokens for the generated summary.
        verbose (bool): If True, print generation details.
    
    Returns:
        str: The generated summary for the chunk.
    """
    # Create a conversation prompt for summarization.
    conversation = [{
        "role": "user", 
        "content": f"Summarize the following section:\n\n{chunk}\n\nSummary:"
    }]
    
    # Apply the chat template to create the full prompt.
    prompt = tokenizer.apply_chat_template(
        conversation=conversation, add_generation_prompt=True
    )
    
    # Generate the summary using mlx-lm's generate function.
    response = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=verbose,
    )
    return response.strip()



In [58]:
def summarize_chunk(chunk, model, tokenizer, max_tokens=max_gen_tokens_exec, verbose=verbose):
    """
    Summarize a given text chunk using the Mistral model via mlx-lm.
    
    Args:
        chunk (str): The text chunk to summarize.
        model: The loaded Mistral model.
        tokenizer: The tokenizer for the Mistral model.
        max_tokens (int): Maximum tokens for the generated summary.
        verbose (bool): If True, print generation details.
    
    Returns:
        tuple: (generated summary for the chunk, human-readable prompt used)
    """
    conversation = [{
        "role": "user", 
        "content": f"Summarize the following text:\n\n{chunk}\n\nSummary:"
    }]
    
    # Create the prompt using the chat template.
    prompt = tokenizer.apply_chat_template(conversation=conversation, add_generation_prompt=True)
    
    # If the prompt is tokenized (e.g. a list of token ids), decode it to a string.
    if not isinstance(prompt, str):
        prompt = tokenizer.decode(prompt)
    
    response = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=verbose,
    )
    summary = response.strip()
    return summary, prompt

def summarize_section_with_context(section_name, chunk, exec_summary, model, tokenizer, max_tokens=max_gen_tokens_other, verbose=verbose):
    """
    Summarize a chunk of a section using the executive summary as context.
    
    Args:
        section_name (str): The name of the section being summarized.
        chunk (str): The chunk of text from this section.
        exec_summary (str): The summary of the executive summary.
        model: The loaded Mistral model.
        tokenizer: The tokenizer for the Mistral model.
        max_tokens (int): Maximum tokens for the generated summary.
        verbose (bool): If True, print generation details.
    
    Returns:
        tuple: (generated summary for the chunk, human-readable prompt used)
    """
    prompt_text = (
        f"This is the {section_name} section of a business case summarized as: '{exec_summary}'.\n\n"
        f"Please summarize the following text:\n\n{chunk}\n\nSummary:"
    )
    conversation = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(conversation=conversation, add_generation_prompt=True)
    
    # Decode the prompt if necessary.
    if not isinstance(prompt, str):
        prompt = tokenizer.decode(prompt)
    
    response = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=verbose,
    )
    summary = response.strip()
    return summary, prompt


In [61]:
output_file = "layered_summaries_with_prompts_streamed.jsonl"
# Set generation parameters
max_gen_tokens_exec = 500   # For the executive summary (layer 1)
max_gen_tokens_other = 1000  # For the other sections (layer 2)
verbose = True

# Define the set of keys that represent the executive summary
exec_keys = {"EXECUTIVE SUMMARY", "PROJECT EXECUTIVE SUMMARY"}

# Open the output file for writing
with open(output_file, "w") as outfile:
    # Process each record in the combined data
    for record in data:
        filename = record.get("filename", "unknown_file")
        sections = record.get("sections", {})
        section_summaries = {}  # To store final overall summary per section
        section_prompts = {}    # To store a list of prompts used for each section

        # --- Step 1: Find and summarize the Executive Summary ---
        exec_summary = None
        for key, text in sections.items():
            if key.upper() in exec_keys:
                exec_chunks = chunk_text(text, max_tokens=1000, overlap=100)
                exec_chunk_summaries = []
                exec_chunk_prompts = []
                for chunk in exec_chunks:
                    summary, prompt = summarize_chunk(chunk, model, tokenizer, max_tokens=max_gen_tokens_exec, verbose=verbose)
                    exec_chunk_summaries.append(summary)
                    exec_chunk_prompts.append(prompt)
                exec_summary = " ".join(exec_chunk_summaries)
                # Store using the original key.
                section_summaries[key] = exec_summary
                section_prompts[key] = exec_chunk_prompts
                break  # Use the first matching executive summary found

        if exec_summary is None:
            exec_summary = "No executive summary available."

        # --- Step 2: Summarize the other sections using the executive summary as context ---
        for section_name, text in sections.items():
            if section_name.upper() in exec_keys:
                continue  # Skip executive summary sections
            if text:
                chunks = chunk_text(text, max_tokens=1000, overlap=100)
                chunk_summaries = []
                chunk_prompts = []
                print(f"Processing '{section_name}' in file {filename}: {len(chunks)} chunk(s)")
                for chunk in chunks:
                    summary, prompt = summarize_section_with_context(
                        section_name, chunk, exec_summary, model, tokenizer, max_tokens=max_gen_tokens_other, verbose=verbose
                    )
                    chunk_summaries.append(summary)
                    chunk_prompts.append(prompt)
                overall_summary = " ".join(chunk_summaries)
                section_summaries[section_name] = overall_summary
                section_prompts[section_name] = chunk_prompts
            else:
                section_summaries[section_name] = "No content available."
                section_prompts[section_name] = []

        # Build the result for the current record.
        result_record = {
            "filename": filename,
            "summaries": section_summaries,
            "prompts": section_prompts
        }
        # Write the result to the output file.
        outfile.write(json.dumps(result_record) + "\n")
        # Flush to ensure it's written to disk and memory is freed.
        outfile.flush()

print(f"Layered summaries and prompts generated and saved to {output_file}")


The Home Upgrade Grant Phase 2 (HUG2) is a project aimed at improving the energy performance and heating systems of off-gas grid homes in England, with a focus on the West Midlands region. The project is led by Matthew Eccles and is part of a larger consortium called the Midlands Net Zero Hub (MNZH), which was awarded a total of £75.2 million by the Department of Business, Energy and Industrial Strategy (BEIS). The West Midlands Combined Authority (WMCA) received a provisional allocation of £16,082,000 for retrofitting domestic houses in the West Midlands.

The project aims to retrofit at least 813 off-gas grid homes, with each home retrofit package not exceeding £18,000. The WMCA's strategy is to cut regional carbon emissions to net zero by 2041, and the project is a step towards achieving this goal. However, the market for domestic energy efficiency and retrofit remains under-developed, with low demand and capacity in the supply chain.

To help overcome these challenges, the WMCA has

In [60]:
results


[{'filename': '[11]wmca-saf-fbc-v3-2 CW Gigapark Sep 2024.md',
  'summaries': {},
  'prompts': {}},
 {'filename': '[14]20230123 HUG2 BJC Observation Report v1.1.md',
  'summaries': {'PROJECT EXECUTIVE SUMMARY': "The Home Upgrade Grant Phase 2 (HUG2) is a project aimed at improving the energy performance and heating systems of off-gas grid homes in England, with a focus on the West Midlands region. The project is led by Matthew Eccles and is part of a larger consortium called the Midlands Net Zero Hub (MNZH), which was awarded a total of £75.2 million by the Department of Business, Energy and Industrial Strategy (BEIS). The West Midlands Combined Authority (WMCA) received a provisional allocation of £16,082,000 for retrofitting domestic houses in the West Midlands.\n\nThe project aims to retrofit at least 813 off-gas grid homes, with each home retrofit package not exceeding £18,000. The WMCA's strategy is to cut regional carbon emissions to net zero by 2041, and the project is a step to

In [62]:
import gc
gc.collect()

5484